<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB_WEEK2_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주제** 전기차 가격 예측 해커톤

https://dacon.io/competitions/official/236424/codeshare/12180?page=1&dtype=recent

**코드 흐름**

**전처리**

결측치 처리 assign 이용: '배터리용량'의 결측치에 평균값 대치

범주형 데이터 인코딩: LabelEncoder() 이용

특성 스케일링: StandardScaler() 이용. train, test 모두에 적용.

**모델 학습**

하이퍼파라미터 튜닝: GridSearchCV 이용.

위에서 얻은 하이퍼파라미터를 이용해 모델 학습 후 평가: 훈련 데이터 RMSE를 구함.

테스트 데이터 예측 수행.

**새롭게 알게 된 점/어려운 점/배울 점**

학습 이전 과정에서 시각화 자료가 없어 코드만 보고 내용을 판단하기 어려웠다. 필요한 코드들만 정리되어 있었으나 직접 내가 데이터를 학습할 때는 시각화를 보다 이용해야겠다는 생각을 했다. 하이퍼파라미터 과정에서 gridsearchCV를 이용할 수도 있으나 이는 시간이 오래 걸리는 관계로 다른 하이퍼파라미터 서치 알고리즘을 이용할 수도 있겠다는 것을 배웠다.

In [ ]:
# 결측치 처리
train = train.assign(배터리용량=train['배터리용량'].fillna(train['배터리용량'].mean()))
test = test.assign(배터리용량=test['배터리용량'].fillna(train['배터리용량'].mean()))

In [ ]:
# 범주형 변수 레이블 인코딩
categorical_features = [col for col in x_train.columns if x_train[col].dtype == 'object']

for col in categorical_features:
    le = LabelEncoder()
    x_train[col] = le.fit_transform(x_train[col])
    for case in np.unique(x_test[col]):
        if case not in le.classes_:
            le.classes_ = np.append(le.classes_, case)
    x_test[col] = le.transform(x_test[col])

In [ ]:
# 하이퍼파라미터 서치
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_model = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf_model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)